# Preparing Alpha Factors and Features to predict Stock Returns

In [3]:
import os
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr
from talib import RSI, BBANDS, MACD, ATR

In [4]:
START = '2010-01-01'
END = '2025-12-30'
path_data = os.path.join("..", "..", "data", "assets.h5")

In [5]:
sns.set_style('whitegrid')
idx = pd.IndexSlice

In [8]:
with pd.HDFStore(path_data) as store:
    prices = (store["stocks/chile"]
              .loc[idx[START : END, : ], :]
              .assign(volume = lambda x : x.Volume.div(1000))
              .swaplevel()
              .sort_index()
              )

In [11]:
prices.head()

Price               Open   High  Low  Close      Volume     volume
Ticker Date                                                       
ABC.SN 2024-07-23   9.36  10.00  9.3  9.827  14849370.0  14849.370
       2024-07-24  10.00  10.30  9.5  9.801  25823660.0  25823.660
       2024-07-25   9.90   9.90  9.8  9.800   3695602.0   3695.602
       2024-07-26   9.70   9.92  9.7  9.892   5669528.0   5669.528
       2024-07-29   9.70   9.92  9.7  9.892   5669528.0   5669.528

# Remove stocks with few observations

In [16]:
min_obs = 2 * 20 * 12 # 2 years

n_obs = prices.groupby(level='Ticker').size()
keep = n_obs[n_obs > min_obs].index
prices = prices.loc[idx[keep, :], :]